In [6]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy
import pandas.io.sql as sqlio 
from datetime import datetime
import xlrd
import matplotlib
import matplotlib.pyplot as plt 
import unidecode
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd 
import time 
import requests 

In [7]:
# Configurar el navegador
options = webdriver.ChromeOptions()
options.add_argument('--incognito') # options.add_argument('--headless')  # si quieres ocultar el navegador
service = Service() 
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10) 

In [ ]:
# Define the URL
driver.get("https://datudeli.com/compra-aqui/")
time.sleep(3)

In [9]:
# Esperar a que cargue el menú y capturar todas las categorías
category_links = []
try: 
    submenu = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul.sub-menu")))
    links = submenu.find_elements(By.TAG_NAME, "a")
    for link in links:
        name = link.get_attribute("innerText").strip()
        href = link.get_attribute("href")
        if name and href:
            category_links.append((name, href))
        
except Exception as e: 
    print("No se pudo obtener el menú de categorías:", e)
    driver.quit()

# Inicializar resultados
products = []

# Iterar por cada categoría
for category_name, category_url in category_links:
    print(f"Scrapeando categoría: {category_name}")
    driver.get(category_url)
    time.sleep(3) 

    while True:
        # Analizar el HTML actual
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        items = soup.select("ul.products li.product")

        if not items:
            print(f"No se encontraron productos en la categoría {category_name}")
            break

        for item in items:
            name_elem = item.select_one("h2.woocommerce-loop-product__title")
            price_elem = item.select_one("span.woocommerce-Price-amount")

            product_name = name_elem.get_text(strip=True) if name_elem else "Sin nombre"
            product_price = price_elem.get_text(strip=True) if price_elem else "Sin precio"

            products.append({
                "category": category_name,
                "product_name": product_name,
                "price": product_price
            })

        # Verificar si hay botón "Siguiente"
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, 'a.next.page-numbers')
            driver.execute_script("arguments[0].scrollIntoView(true);", next_btn)
            next_btn.click()
            time.sleep(3)
        except:
            break  # No hay más páginas

driver.quit() 

# Crear DataFrame
df = pd.DataFrame(products) 

# Mostrar resumen 
print(df.head()) 
print(f"Total productos scrapeados: {len(df)}")

Scrapeando categoría: Abarrotes
Scrapeando categoría: Aves
Scrapeando categoría: Carnes importadas
Scrapeando categoría: Certified Angus Beef
Scrapeando categoría: Chocolates italianos
Scrapeando categoría: Embutidos
Scrapeando categoría: Lácteos
Scrapeando categoría: Pastelería
Scrapeando categoría: Pescados y mariscos
Scrapeando categoría: Productos japoneses
Scrapeando categoría: Productos veganos
No se encontraron productos en la categoría Productos veganos
    category                           product_name   price
0  Abarrotes     Aceite de Oliva Italiano Santagata  $19.60
1  Abarrotes           Aceite de Trufa Negra Espora  $15.60
2  Abarrotes     Aceitunas Kalamatas C/Hueso Roland   $9.18
3  Abarrotes  Aceitunas Kalamatas En Mitades Roland   $8.28
4  Abarrotes     Aceitunas Kalamatas S/Hueso Roland   $9.63
Total productos scrapeados: 65


In [ ]:
precios = df.dropna()
precios.to_excel(r"C:\Users\juan.valladares\Documents\Web Scraper\Scraper - Datu Deli/Scrapper Precios con Categoría - Datu Deli Prueba.xlsx", index=False)

: 